In [ ]:
# 1. Imports and simulation parameters

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from order_book_simulator import LOBSimulator, LOBState, LOBPlotter
from lob_optimizer import (
    PathCache, PolicyOptimizer, SellPolicy,
    g_imbalance, g_bid_price, g_microprice, g_ask_pressure,
)

# ── Shared simulation parameters ───────────────────────────────────────────────
sim_kwargs = dict(
    pa=100.005, qa=10,
    pb=100.000, qb=10,
    lambda_a=5.0, lambda_b=5.0,
    lambda_ma=2.0, lambda_mb=2.0,
    theta_a=0.5, theta_b=0.5,
    tick_size=0.005,
)

T       = 50    # time horizon (seconds)
N_PATHS = 500   # Monte Carlo paths

print(f"Simulation parameters: {sim_kwargs}")
print(f"Horizon T={T}, paths={N_PATHS}")


# 2. Problem Statement

## Optimal Stopping for a Single Sell Order

We hold one unit of an asset and must sell it before a hard deadline $T$.
At each LOB event we observe the state $(p_a, q_a, p_b, q_b)$ and choose
whether to sell (market sell at best bid $p_b$) or wait.

### Stopping rule

We parameterise the sell decision by a **linear urgency threshold**:

$$f(t) = c_0 + c_1 \cdot t$$

and a **state signal** $g : \text{LOBState} \to \mathbb{R}$.  We sell at the
first time the signal exceeds the threshold, or at the deadline:

$$\tau = \min\!\bigl(T,\; \inf\{t : g(\text{state}_t) > f(t)\}\bigr)$$

### Objective

Maximise the **expected selling price**:

$$\max_{c_0,\, c_1} \; \mathbb{E}\bigl[p_b(\tau)\bigr]$$

### Signal functions $g$

| Name | Formula | Intuition |
|---|---|---|
| `g_imbalance` | $q_b/(q_a+q_b)$ | high → bid-heavy → upward price pressure |
| `g_bid_price` | $p_b$ | sell when best bid is high |
| `g_microprice` | $I\cdot p_a + (1-I)\cdot p_b$ | Stoikov micro-price |
| `g_ask_pressure` | $q_a/(q_a+q_b)$ | high → ask-heavy → sell now |

### Solution method

1. **Monte Carlo**: pre-simulate $N$ independent LOB paths up to $T$.
2. **Grid scan**: evaluate $\mathbb{E}[p_b(\tau)]$ on a $(c_0, c_1)$ grid.
3. **SGD**: gradient ascent with finite-difference gradient estimates on mini-batches.


In [ ]:
# 3. Build PathCache

print("Simulating paths…  (this may take ~30 s for 500 paths)")
cache = PathCache.build(sim_kwargs, T=T, n_paths=N_PATHS, g=g_imbalance, seed=0)

# Summary statistics
n_events = [len(df) - 1 for df in cache.paths]   # exclude initial row
print(f"\nPathCache built: {cache.n_paths} paths, T={cache.T}")
print(f"Events per path — mean: {np.mean(n_events):.0f}  "
      f"min: {np.min(n_events)}  max: {np.max(n_events)}")
print(f"g function: g_imbalance  (signal = bid-queue fraction)")


In [ ]:
# 4. Benchmark policies
# With g_imbalance ∈ (0, 1):
#   c0 = -inf → threshold always below g → triggers at t=0 → "sell immediately"
#   c0 = +inf → threshold always above g → never triggers → "sell at deadline"

optimizer = PolicyOptimizer(cache)

benchmarks = {
    "Sell immediately"  : (-1e9, 0.0),   # threshold -∞ → always triggers at t=0
    "Sell at deadline"  : (1e9,  0.0),   # threshold +∞ → never triggers early → wait until T
    "Neutral threshold" : (0.5,  0.0),   # sell when imbalance > 0.5
}

bench_df = optimizer.compare(benchmarks)
print("Benchmark policy comparison:")
print(bench_df.to_string(float_format="{:.5f}".format))


In [ ]:
# 5. Grid scan — heatmap of E[pb(τ)] over (c0, c1)

c0_grid = np.linspace(-1.0, 2.0, 30)
c1_grid = np.linspace(-0.05, 0.05, 30)

print("Running grid scan…")
scan_df = optimizer.scan(c0_grid, c1_grid)

# Pivot for heatmap
pivot = scan_df.pivot(index="c1", columns="c0", values="expected_price")

# Best grid point
best_row = scan_df.loc[scan_df["expected_price"].idxmax()]
print(f"Grid best: c0={best_row.c0:.3f}  c1={best_row.c1:.4f}  "
      f"E[price]={best_row.expected_price:.5f}")

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.pcolormesh(
    pivot.columns, pivot.index, pivot.values,
    cmap="RdYlGn", shading="auto",
)
plt.colorbar(im, ax=ax, label="E[pb(τ)]")
ax.scatter([best_row.c0], [best_row.c1], marker="*", s=200,
           color="navy", zorder=5, label=f"Grid max ({best_row.c0:.2f}, {best_row.c1:.4f})")
ax.set_xlabel("c0  (threshold intercept)")
ax.set_ylabel("c1  (threshold slope)")
ax.set_title("Grid Scan: Expected Selling Price E[pb(τ)]  —  g = imbalance")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# 6. SGD optimisation

print("Running SGD (300 steps, batch_size=64)…")
result = optimizer.sgd(
    c0_init=0.5, c1_init=0.0,
    lr=0.02, n_steps=300, batch_size=64,
    epsilon=1e-3, lr_decay=0.997,
)

hist = result["history"]
opt_c0, opt_c1 = result["c0"], result["c1"]
opt_price      = result["expected_price"]

print(f"\nSGD optimum: c0={opt_c0:.4f}  c1={opt_c1:.6f}")
print(f"E[price] = {opt_price:.5f}")
print(f"vs. 'Sell at deadline': {bench_df.loc['Sell at deadline','mean_price']:.5f}")
print(f"vs. 'Neutral threshold': {bench_df.loc['Neutral threshold','mean_price']:.5f}")

# ── Plot 1: convergence ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(hist["step"], hist["batch_price"], alpha=0.5, lw=0.8, color="steelblue",
        label="Mini-batch price")
ax.plot(hist["step"], hist["full_price"],  lw=1.5, color="darkblue",
        label="Full-data E[price]")
ax.axhline(bench_df.loc["Sell at deadline", "mean_price"],
           ls="--", color="orange", lw=1, label="Sell-at-deadline baseline")
ax.set_xlabel("SGD step")
ax.set_ylabel("E[pb(τ)]")
ax.set_title("SGD Convergence")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.25)

# ── Plot 2: SGD trajectory on the heatmap ─────────────────────────────────────
ax2 = axes[1]
im2 = ax2.pcolormesh(
    pivot.columns, pivot.index, pivot.values,
    cmap="RdYlGn", shading="auto", alpha=0.8,
)
plt.colorbar(im2, ax=ax2, label="E[pb(τ)]")

# Trajectory coloured by step (light → dark)
sc = ax2.scatter(
    hist["c0"], hist["c1"],
    c=hist["step"], cmap="Blues", s=8, zorder=4, alpha=0.8,
)
plt.colorbar(sc, ax=ax2, label="SGD step")
ax2.scatter([opt_c0], [opt_c1], marker="*", s=250, color="navy",
            zorder=6, label=f"SGD opt ({opt_c0:.2f}, {opt_c1:.4f})")
ax2.scatter([0.5], [0.0], marker="o", s=80, color="black",
            zorder=6, label="Start (0.5, 0.0)")
ax2.set_xlabel("c0")
ax2.set_ylabel("c1")
ax2.set_title("SGD Trajectory on Grid-Scan Landscape")
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# 7. Sensitivity: different g functions

G_FUNCTIONS = {
    "g_imbalance"   : g_imbalance,
    "g_bid_price"   : g_bid_price,
    "g_microprice"  : g_microprice,
    "g_ask_pressure": g_ask_pressure,
}

results_g: dict[str, dict] = {}

for gname, gfn in G_FUNCTIONS.items():
    print(f"  {gname}: building cache…", end=" ", flush=True)
    c_i = PathCache.build(sim_kwargs, T=T, n_paths=N_PATHS, g=gfn, seed=42)
    opt_i = PolicyOptimizer(c_i)
    print("running SGD…", end=" ", flush=True)
    res_i = opt_i.sgd(
        c0_init=0.5, c1_init=0.0,
        lr=0.02, n_steps=150, batch_size=64,
        epsilon=1e-3, lr_decay=0.995,
    )
    results_g[gname] = res_i
    print(f"done — E[price]={res_i['expected_price']:.5f}  "
          f"c0={res_i['c0']:.3f}  c1={res_i['c1']:.5f}")

# Bar chart of optimal E[price] per g function
names  = list(results_g.keys())
prices = [results_g[n]["expected_price"] for n in names]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(names, prices, color=["steelblue", "salmon", "mediumseagreen", "darkorange"])
ax.bar_label(bars, fmt="{:.5f}", fontsize=9, padding=3)
ax.set_ylabel("Optimal E[pb(τ)]")
ax.set_title("Optimal Expected Price by Signal Function  (SGD, 150 steps)")
ax.set_ylim(min(prices) - 0.005, max(prices) + 0.01)
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# 8. Visualise a sample path under the optimal policy

# Pick a representative path (e.g. path index 0)
PATH_IDX = 0
df_path  = cache.paths[PATH_IDX]

t_vals  = df_path["time"].values
g_vals  = df_path["g_val"].values
pb_vals = df_path["pb"].values
f_vals  = opt_c0 + opt_c1 * t_vals   # urgency threshold

# Find stopping time τ for this path
triggered = g_vals > f_vals
if triggered.any():
    tau_idx  = int(np.argmax(triggered))
    sell_reason = "signal crossed threshold"
else:
    tau_idx  = len(df_path) - 1
    sell_reason = "deadline reached"

tau       = t_vals[tau_idx]
sell_price = pb_vals[tau_idx]

print(f"Path {PATH_IDX}: τ = {tau:.2f}  ({sell_reason})  pb(τ) = {sell_price:.5f}")

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# Top panel: bid price dynamics
ax_top = axes[0]
plotter = LOBPlotter(df_path, tick_size=sim_kwargs["tick_size"])
plotter.plot_price_dynamics(ax=ax_top)
ax_top.axvline(tau, color="purple", lw=1.8, ls="--", label=f"τ = {tau:.1f}")
ax_top.scatter([tau], [sell_price], s=120, color="purple",
               zorder=6, label=f"Sell at pb={sell_price:.4f}")
ax_top.legend(fontsize=8, ncol=5)
ax_top.set_title(f"Path {PATH_IDX}: Price Dynamics  (optimal policy  c0={opt_c0:.3f}, c1={opt_c1:.4f})")

# Bottom panel: g(state) and f(t) = c0 + c1*t
ax_bot = axes[1]
ax_bot.step(t_vals, g_vals, where="post", color="darkorange", lw=0.9, alpha=0.85,
            label="g(state) = imbalance")
ax_bot.plot(t_vals, f_vals, color="black", lw=1.4, ls="--",
            label=f"f(t) = {opt_c0:.3f} + {opt_c1:.4f}·t")
ax_bot.axvline(tau, color="purple", lw=1.8, ls="--", label=f"τ = {tau:.1f}")
ax_bot.scatter([tau], [g_vals[tau_idx]], s=120, color="purple", zorder=6)
ax_bot.set_xlabel("Time")
ax_bot.set_ylabel("Signal / Threshold")
ax_bot.set_title("Signal g(state) vs Urgency Threshold f(t)")
ax_bot.legend(fontsize=8)
ax_bot.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()
